# Test Anatomical Validity Detection

`AnatomicalValidity` is a class (`utils/anatomical_validity.py`) that goes through a pre-defined series of criteria to determine if a given mask is anatomically valid. For example, a cardiac mask should have exactly 1 right ventricle, 1 left ventricle and 1 myocardium.

This notebook tests whether the detector is working as expected.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 1 level deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ..

assert os.getcwd().endswith(root_dir)

Preliminary script to load data and setup device.

In [ ]:
import lightning as L

from const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

## Checking Test Data

We'd expect the test data to be anatomically valid... right?

Unfortunately, this isn't the case. There are a few caveats, and this is due to resolution downsampling in our preprocessing pipeline: the original masks are around ${250 \times 250}$, and we downsample to ${128 \times 128}$ with nearest neighbour interpolation. This means we lose some fine detail, and it can lead to anatomically invalid masks (as determined by the criteria) at the pixel-wise level.

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader()
data_test = get_data(loader_test)
data_test.shape

Out of 1076 test masks, 3 are invalid.

In [ ]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker

invalid_masks_idx = []

for i, mask in enumerate(data_test):
    AV = AnatomicalValidityChecker(mask)
    anatomical_violations = AV.count_violations()
    
    if anatomical_violations > 0:
        invalid_masks_idx.append(i)
        print(f"Anatomical violations in mask {i}: {anatomical_violations}")

Let's take a look at these 3 masks. You should see that there are 2 right ventricles in each of them. The extra right ventricle is only a few pixels large, likely caused by the downsampling.

In [ ]:
import torch
from utils.utils import show_samples

for i in invalid_masks_idx:
    mask = data_test[i]
    
    AV = AnatomicalValidityChecker(mask)
    print(AV.summarise_conditions())
    
    sample = mask.unsqueeze(0)
    sample: torch.Tensor = torch.argmax(sample, dim=1).unsqueeze(1)
    show_samples(sample, rgb=False)

## Test Robustness: Creating Invalid Masks

Now, let's manually disrupt the test data by crafting some invalid masks, to see if the detector catches them.

In [ ]:
def show_mask(mask):
    sample = mask.unsqueeze(0)
    sample: torch.Tensor = torch.argmax(sample, dim=1).unsqueeze(1)
    show_samples(sample, rgb=False, ncol=1, figsize=(2, 2))

mask = data_test[121]
show_mask(mask)

First, let's test hole detection.

From the print logs, we will see that the detector is very robust: it also catches that the left ventricle touches the background (due to the hole), and so there are actually 2 violations.

In [ ]:
# Normal test
mask_with_hole = mask.clone()
mask_with_hole[:, 70:80, 70:80] = 0 # All classes
mask_with_hole[0, 70:80, 70:80] = 1 # BG
show_mask(mask_with_hole)

AV = AnatomicalValidityChecker(mask_with_hole)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

# Extreme test
mask_with_hole = mask.clone()
mask_with_hole[:, 70:71, 70:71] = 0 # All classes
mask_with_hole[0, 70:71, 70:71] = 1 # BG
show_mask(mask_with_hole)

AV = AnatomicalValidityChecker(mask_with_hole)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

Now, let's create a mask with 2 of each components, by slicing the mask in half. Obviously, this also induces other violations, like LV touching background.

In [ ]:
from const import ACDC
from utils.utils import acdc_class_id_to_idx

# Normal test
sliced_mask = mask.clone()
sliced_mask[:, 70:80, :] = 0 # All classes
sliced_mask[0, 70:80, :] = 1 # BG
show_mask(sliced_mask)

AV = AnatomicalValidityChecker(sliced_mask)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

# Extreme test
sliced_mask = mask.clone()
sliced_mask[:, 70:71, :] = 0 # All classes
sliced_mask[0, 70:71, :] = 1 # BG
# For the extreme test, let's reconnect the LV and see if the dectector can
# detect that there is only 1 LV
sliced_mask[acdc_class_id_to_idx(ACDC.ClassLabel.LV), 70:71, 70:71] = 1
sliced_mask[0, 70:71, 70:71] = 0
show_mask(sliced_mask)

AV = AnatomicalValidityChecker(sliced_mask)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

For good measure, let's take the extreme test further by stitching a 1-pixel hole in the LV, where the boundary of the hole is 1 pixel thick.

In [ ]:
sliced_mask[acdc_class_id_to_idx(ACDC.ClassLabel.LV), 70:71, 72:73] = 1
sliced_mask[0, 70:71, 72:73] = 0
show_mask(sliced_mask)

AV = AnatomicalValidityChecker(sliced_mask)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

Now, make the left ventricle touch the right ventricle.

In [ ]:
# Normal test
mask_big_lv = mask.clone()
mask_big_lv[:, 70:80, :] = 0 # All classes
mask_big_lv[acdc_class_id_to_idx(ACDC.ClassLabel.LV), 70:80, :] = 1
show_mask(mask_big_lv)

AV = AnatomicalValidityChecker(mask_big_lv)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

# Extreme test: Replace 1 pixel of the RV with LV
mask_big_lv = mask.clone()
mask_big_lv[:, 30:31, 30:31] = 0 # All classes
mask_big_lv[acdc_class_id_to_idx(ACDC.ClassLabel.LV), 30:31, 30:31] = 1
show_mask(mask_big_lv)

AV = AnatomicalValidityChecker(mask_big_lv)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")

For the final criteria, it's difficult to modify the mask to separate the RV from the MYO, so let's hand-pick a generated mask that violates this criteria.

In [ ]:
from arch.nvae.nvae import NVAE

model_path = "logs/nvae_acdc/latent-skip/pc-4-ws-6420-b-1/checkpoints/epoch=98-step=21186.ckpt"

# Load model
model = NVAE.load_from_checkpoint(model_path)

# Seed
L.seed_everything(SEED)

with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_fake = model.decoder.generate(num_samples=200, device=device)
    feats_fake = model.conditional_coder(x_fake)

# Discretise probabilistic map then view generations
generations = torch.argmax(feats_fake, dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=20, figsize=(10, 20))

This mask violates the criteria we want.

In [ ]:
show_samples(generations[177], rgb=False, ncol=1, figsize=(2, 2))

In [ ]:
from utils.utils import discretise

sample_feats = feats_fake[177].unsqueeze(0)
discretised_feats = discretise(sample_feats).cpu().squeeze(0)

AV = AnatomicalValidityChecker(discretised_feats)
print(AV.summarise_conditions())
print(f"Anatomical violations: {AV.count_violations()}")